In [0]:
%sql

-- *** Window Functions ***

-- Window Functions Synatx 
-- Window Function  [Partition Clause , Order Clause , Frame Clause]
-- AVG(Sales) OVER (PARTITION BY Region ORDER BY Sales ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT)


-- Perform Calclutions (Aggregations) on specifc subset of data without lossing the level of detials
-- Perform Calclutions across rows related to current row
-- Keep all rows (unlike Group by)
-- Analyzing while preserving all rows (details)
-- Window Functions are applied to a group of rows called a Window
-- Window is defined by PARTITION BY and ORDER BY
-- PARTITION BY defines the groups of rows
-- ORDER BY defines the order of rows within each group
-- Window Functions are applied to each row in the group


----------------------------------------------------------------------------------------------

-- Row_Number() is a Window Function
-- Assigns a unique number to each row within a group
-- The numbering starts at 1 for each group
-- The numbering is sequential and incremented by 1 for each row within the group
-- The numbering is reset to 1 for each new group


----------------------------------------------------------------------------------------------

--  RANK() :    SAME RANKING FOR TIES , GPAS AFTER 
--  DENSE_RANK() :  RANKING WITHOUT GAP FOR TIES 
-- LAG() :  PREVIOUS ROW VALUE
-- LEAD() :  NEXT ROW VALUE
-- NTILE() :  DIVIDE INTO GROUPS
-- CUME_DIST() :  CUMULATIVE PERCENTAGE
-- PERCENT_RANK()



/*

Window Expression Arguments
These categories define what goes inside the parentheses of the function before the OVER clause.

# Empty
Description: No arguments are needed because the function operates based solely on the row's position or the window definition.

Example: RANK() OVER (ORDER BY OrderDate)

# Column
Description: The function performs a calculation on a specific column/field.

Example: AVG(Sales) OVER (ORDER BY OrderDate)

# Number

Description: Certain functions require a specific numeric value to define how they operate (like buckets or offsets).

Example: NTILE(2) OVER (ORDER BY OrderDate) (Note: The image has a small typo "NTEIL", it is correctly spelled NTILE).

# Multiple Arguments

Description: Some advanced functions need several pieces of information (Column, Offset, and Default Value).

Example: LEAD(Sales, 2, 10) OVER (ORDER BY OrderDate)

# Conditional Logic

Description: You can wrap a CASE WHEN statement inside the function to only aggregate data that meets specific criteria.

Example: SUM(CASE WHEN Sales > 100 THEN 1 ELSE 0 END) OVER (ORDER BY OrderDate)
 
 */


/* 

Difference between Group By and Window Functions : 

1. GROUP BY Functions
These are Aggregate Functions used when you want to collapse multiple rows into a single summary row.

COUNT(expr): Counts the number of rows.

SUM(expr): Calculates the total sum of a numeric column.

AVG(expr): Calculates the average value.

MIN(expr): Finds the minimum value in the set.

MAX(expr): Finds the maximum value in the set.

2. WINDOW Functions
These functions allow you to perform calculations while preserving the individual rows. They are categorized into three main types:

A. Aggregate Functions (Used as Windows)
These are the same as the Group By functions, but when used with the OVER() clause, they don't collapse the rows.

COUNT(expr)

SUM(expr)

AVG(expr)

MIN(expr)

MAX(expr)

B. Rank Functions
These are used to assign a ranking or a number to each row within a partition.

ROW_NUMBER(): Assigns a unique sequential integer to rows.

RANK(): Assigns a rank with gaps (e.g., 1, 2, 2, 4).

DENSE_RANK(): Assigns a rank without gaps (e.g., 1, 2, 2, 3).

CUME_DIST(): Calculates the cumulative distribution of a value.

PERCENT_RANK(): Calculates the relative rank of a row as a percentage.

NTILE(n): Divides rows into n number of ranked groups (tiles).

C. Value (Analytics) Functions
These allow you to compare values across different rows in the same result set.

LEAD(expr, offset, default): Accesses data from a subsequent row (the "next" row).

LAG(expr, offset, default): Accesses data from a previous row (the "prior" row).

FIRST_VALUE(expr): Returns the value from the first row in the window.

LAST_VALUE(expr): Returns the value from the last row in the window. 

*/


In [0]:
%sql
-- 

select order_id,order_date, rank() over (order by order_date) as rank_num
 from orders


In [0]:
%sql

-- Show frist product bought by each customer

select * from 

(select c.customer_id, 
c.first_name, 
od.product_name, 
od.product_id,
row_number() over (partition by c.customer_id order by o.order_date) as Purchase_Sequence
from customers c
right join orders o on c.customer_id = o.customer_id
join order_details od on o.order_id = od.order_id  
)

where Purchase_Sequence= 1


In [0]:
%sql
select product_name , quantity ,
rank() over (order by quantity desc) as ranking
from order_details 


In [0]:
%sql
-- Assign bonus based on best-selling products in each order 

select  order_id,
 product_name,
quantity,
rank() over (partition by order_id  order by quantity desc) as ranking
from order_details

In [0]:
select 
product_name,
quantity,
dense_rank() over (order by quantity desc) as dens_rank,
rank() over (order by quantity desc) as rank


from order_details

In [0]:
-- lable products popularity tiers based on quntity

select
product_name ,
sum(quantity) as Total ,
dense_rank() over (order by sum(quantity) desc) as rank
from order_details
group by product_name




In [0]:
-- LAG ()   Compar with previous row
-- LEAD ()  compare with next row

select 
order_id,
order_date ,
LEAD(order_date) over (order by order_date) as Next_order_date 
from orders

/*
select 
order_id,
order_date ,
LAG(order_date) over (order by order_date) as previos_order_date 
from orders
*/



In [0]:
-- Analayz the customers trends #1 (period from last order)

select 
o.customer_id ,
o.order_id,
o.total_amount,
o.order_date,
LAG(o.order_date) over (partition by o.customer_id order by o.customer_id) as Previous_order_date
from orders o



In [0]:
-- Analayz the customers trends #1 (period from last order)
-- Analayz the customers trends #2 (did they buy more or less)

select 
o.customer_id ,
o.order_id,
o.order_date,
LAG(o.order_date) over (partition by o.customer_id order by o.customer_id) as Previous_order_date ,
o.total_amount,
LAG(o.total_amount) over (partition by o.customer_id order by o.customer_id) as previous_amount,
(o.total_amount-previous_amount) as difference
from orders o
-- order by order_date asc



In [0]:
-- SUM() --->   Running Total (qauntity) of Cummulative Sum

select order_id , product_name , quantity ,
sum(quantity) over (order by order_id) as total
from order_details
 

In [0]:
-- SUM() --->   Running Total (salary) of Cummulative Sum
select 
first_name,
department,
salary ,
sum(salary) over (partition by department order by emp_id) as running_total
from employees

/*
select 
first_name,
department,
salary ,
sum(salary) over (partition by department order by emp_id) as running_total
from employees
*/

In [0]:
/*

                                    *** Normalization*** 

The process of organizing data in a database to reomve redundancy and improve data integrity
  Goal :  
  1.Store data once and reference it many times (Link it properly)
  2.Ensure data entity + efficient storage
  3.Acheied via splitting data into multiple tables & defining relationship

  - Too mach normalization --> many jion --> slower qeuries
  - Trade-off :
    1.OLTP  -> NORMLIZE FRO INTEGRITY
    2.OLAP  -> SOMETIMES DENORMALIZE FOR PERFORMANCE


*/

In [0]:
/*
                                         ***   SQL Optimization  ***

SQL optimization is the process of improving the speed and efficiency of database queries so they return accurate results while using minimal computational resources such as CPU, memory, and disk I/O. It includes both automated processes within the database engine and manual techniques applied by developers to refine query logic.

EXPLAIN PLAN :

1. Show EXECUTION Plan 
2. Helps find full Scans , index use & join strategy
3. Crucial for debuging Performance

# NOTE ...
 
* BAD JOINS  =
1.(row explosion/duplicates), 
2.slow performance, 
3.misleading analytics

* NO fillter = FULL TABLE SCAN
* Missing Indexes = No shortcuts for lookup

----------------------------------------------------------------------------------------------------------------

$$$ Core Components

- Query Optimizer
A built-in component of a database management system (DBMS) that analyzes SQL statements and determines the most efficient way to execute them.

- Execution Plan
A detailed roadmap generated by the optimizer that shows how the database will retrieve data, including index usage and join order.

- Statistics
Metadata about data distribution and volume that helps the optimizer estimate the cost of different execution strategies.

----------------------------------------------------------------------------------------------------------------
# Essential Optimization Techniques

1.Strategic Indexing
Create indexes on columns frequently used in WHERE, JOIN, and ORDER BY clauses to avoid full table scans.

2.Selective Column Retrieval
Avoid SELECT *; instead, retrieve only the necessary columns to reduce data processing and transfer.

3.Efficient Filtering
Use WHERE instead of HAVING whenever possible to filter rows before aggregation.

4.Join Optimization
Prefer joins over unnecessary subqueries and ensure join columns have matching data types to avoid implicit conversions.

5.Limiting Result Sets
Use LIMIT or TOP to restrict the number of rows returned, especially during testing or previewing.

------------------------------------------------------------------------------------------------------------------
Why It Matters  :

- Performance
Reduces query execution time and improves responsiveness for users.

- Cost Control
Efficient queries lower compute usage, which is especially important in cloud-based systems.

- Scalability
Optimized queries allow databases to handle more users and larger workloads.

- Reliability
Prevents slowdowns, locking issues, and system instability caused by inefficient queries.

----------------------------------------------------------------------------------------------------------------

*** Clustered   Index ***   : 
Physically sorts the table based on the indexed column, for improving data retrieval speed.

. Only one Clustered Index per Table
. Faster Speed when Filtering by index Column
. Slower Speed when Updating Table


*** Non-Clustered Index *** : 
Stores a copy of indexed columns along with a pointer to the actual data, allowing faster lookups without altering data order.

. Seprate index Strucure from Table (dose not sort the table)
. Can have multiple per Table
. Points to row loaction in table






------------------------------------------     Optimizations Tips & Technique         -----------------------------------

1.   Avoid fUll Table Scan  :
SELECT * FROM table;  -- Select only What you need
SELECT column1, column2 FROM table; -- Select only needed columns

2.   WHERE Before Group BY :  (Smaller data set grouped = faster)
SELECT column1, SUM(column2) FROM table GROUP BY column1; -- Avoid this
SELECT column1, SUM(column2) FROM table WHERE column1 = 'value'

3.   Use Indexes : 
    . CREATE INDEX index_name ON table (column)

4. Avoid Distinct when possible : (Needs to Sort the data)
  SELECT DISTINCT first_name FROM  customers
   .  Validate Data at source 
   .  Better Design Schema to aviod dupliacte

SELECT DISTINCT column1 FROM table; -- Avoid this
SELECT column1 FROM table GROUP BY column1

5. Partitionnig :     (Split your table into pieces)
  . Physicaly split your data into smaller pieces based on a column
  . each query only read the relevant partition  
      CREATE TABLE table PARTITIONED BY (column);

6. Use EXPLAIN PLAN : 
  . SHOW CREATE TABLE table; -- Show table structure
  . EXPLAIN SELECT ...

7. Bucketing : Divide data into equlal buckets based on a column value  
   . evenly split data based of the hash of the column  
   . improve large join performance
   . CREATE TABLE table (column1, column2) WITH (bucketed_by = ARRAY['column1) 


*/




In [0]:

--  TASKS : 

-- 1.  Assign a rank of employees based on their salary within each department
select first_name , department , salary ,
rank() over(partition by department order by salary desc) as rank_salary from employees 

/*

-- 2.  finding running total of salary within each department
select first_name,department ,salary ,
sum(salary) over(partition by department orderby salary desc) as runnig_salary from employees 

*/



/*


***  Benefits of Self joins 

1. Handle hierarchical data (parent–child relationships)
Very common use case.

Example: employees table

Each employee has a manager_id pointing to another employee
SELECT e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m
ON e.manager_id = m.id;

👉 Lets you see relationships within the same table

2. Compare rows within the same table
You can compare one row to another.

Example:

Find employees with the same salary
SELECT a.name, b.name, a.salary
FROM employees a
JOIN employees b
ON a.salary = b.salary AND a.id <> b.id;

3. Find duplicates
Useful when you don’t have constraints.

SELECT a.*
FROM users a
JOIN users b
ON a.email = b.email AND a.id <> b.id;


--------------------------------------------------------------------------------------

***   Finding a running total  


1. Track trends over time
Instead of just seeing daily numbers, you see the progression.
Example: sales per day vs total sales so far → you can quickly spot growth or slowdowns.

2. Better decision-making
Running totals help answer questions like:

“Have we hit our monthly target yet?”
“At which point did revenue cross a threshold?”

3. Simplifies reporting
Many reports (finance, inventory, analytics) rely on cumulative values:

Total revenue so far
Total users acquired
Total expenses

4. Detect anomalies
Sudden jumps or flat lines in a running total are easier to notice than in raw data.

5. Useful for comparisons
You can compare:

Current cumulative value vs previous periods
Actual vs target progress

*/


